# 11. Baseline Models â€” Modular Training & Evaluation
Refactored notebook with dedicated model-construction cells (Section B) and dedicated per-model training cells (Section C).

- **Models:** `CNN1D`, `LSTM`, `BiLSTM`, `BiLSTM_CNN`, `Classical_LR`
- **Preprocessing Branches:** `gsn_minmax`, `gsn_zscore`, `tbsn_minmax`, `tbsn_zscore`
- **Split:** `trial_split` (reproducible trial-wise 70% TRAIN / 15% VALIDATION / 15% TEST split)
- **Input Contract:** `751` time steps Ã— `180` sensor features (excluding `frame` and `time`)
- **Target:** Walking UPDRS-III score (`label_updrs_iii_walking`, classes `0, 1, 2`)

---
## Section A â€” Setup and Shared Utilities
### Cell 1: Imports, Paths, Configuration, and Device

In [1]:
# ============================================================================
# CELL 1: IMPORTS, PATHS, CONFIGURATION, AND DEVICE
# ============================================================================
import sys, os, json, yaml, hashlib, warnings, re, time, pickle, shutil, copy, uuid, random
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as TF
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    mean_absolute_error,
    confusion_matrix,
    classification_report,
    log_loss,
)
warnings.filterwarnings('ignore')

# â”€â”€ Project root discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'configs').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.models.baselines import (
    CNN1DBaseline,
    LSTMBaseline,
    BiLSTMBaseline,
    BiLSTM_CNN_Baseline,
    extract_statistical_features,
)
from src.evaluation.classification import compute_classification_metrics

# â”€â”€ Output directories â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
for d in ['results/tables', 'results/predictions', 'results/checkpoints']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# â”€â”€ Load YAML configurations â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
with open(PROJECT_ROOT / 'configs' / 'data_config.yaml', 'r') as _f:
    data_cfg = yaml.safe_load(_f)
with open(PROJECT_ROOT / 'configs' / 'experiment_config.yaml', 'r') as _f:
    exp_cfg = yaml.safe_load(_f)

# â”€â”€ Authoritative manifest and constants â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MANIFEST_PATH = PROJECT_ROOT / 'results' / 'tables' / 'window_manifest.csv'
TRIAL_SPLIT_MANIFEST_PATH = PROJECT_ROOT / 'data' / 'metadata' / 'trial_split_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH)

W = int(data_cfg['windowing']['window_size'])          # 751 time frames
F = 180                                                # 180 sensor features
FEAT_COUNT = 180                                       # explicit alias
NUM_CLASSES = int(exp_cfg['evaluation']['num_classes']) # 3 classes (0, 1, 2)
TARGET_COL = exp_cfg['evaluation']['target_column']     # label_updrs_iii_walking
SEED = 42

EXPECTED_MODELS   = ["CNN1D", "LSTM", "BiLSTM", "BiLSTM_CNN", "Classical_LR"]
EXPECTED_BRANCHES = ["gsn_minmax", "gsn_zscore", "tbsn_minmax", "tbsn_zscore"]
EXPECTED_FOLDS    = ["trial_split"]

MAX_EPOCHS_DEFAULT       = 100
EARLY_STOPPING_PATIENCE  = 10
BILSTM_CNN_MAX_EPOCHS    = 400
BILSTM_CNN_PATIENCE      = 10

# â”€â”€ Explicit retrain switches â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Default True for BiLSTM-CNN so stale checkpoints never silently skip training
FORCE_RETRAIN_BILSTM_CNN   = True
FORCE_RETRAIN_OTHER_MODELS = True

# â”€â”€ Session registries (updated as each model training cell completes) â”€â”€â”€â”€â”€â”€â”€â”€
SESSION_MODEL_RESULTS     = {}
SESSION_MODEL_HISTORY     = {}
SESSION_MODEL_PREDICTIONS = {}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=" * 72)
print("SETUP & CONFIGURATION SUMMARY")
print("=" * 72)
print(f"Project root              : {PROJECT_ROOT}")
print(f"Device                    : {device}")
print(f"Random seed               : {SEED}")
print(f"Sequence length (W)       : {W}")
print(f"Input feature count (F)   : {FEAT_COUNT}")
print(f"Target column             : {TARGET_COL} ({NUM_CLASSES} classes)")
print(f"Expected models           : {EXPECTED_MODELS}")
print(f"Expected branches         : {EXPECTED_BRANCHES}")
print(f"Expected split            : {EXPECTED_FOLDS}")
print(f"FORCE_RETRAIN_BILSTM_CNN  : {FORCE_RETRAIN_BILSTM_CNN}")
print("=" * 72)


SETUP & CONFIGURATION SUMMARY
Project root              : C:\Users\lab\Minor_project\Parkinson_gait
Device                    : cuda
Random seed               : 42
Sequence length (W)       : 751
Input feature count (F)   : 180
Target column             : label_updrs_iii_walking (3 classes)
Expected models           : ['CNN1D', 'LSTM', 'BiLSTM', 'BiLSTM_CNN', 'Classical_LR']
Expected branches         : ['gsn_minmax', 'gsn_zscore', 'tbsn_minmax', 'tbsn_zscore']
Expected split            : ['trial_split']
FORCE_RETRAIN_BILSTM_CNN  : True


### Cell 2: Data and Manifest Validation

In [2]:
# ============================================================================
# CELL 2: DATA AND MANIFEST VALIDATION
# ============================================================================
print("=" * 72)
print("DATA AND MANIFEST VALIDATION")
print("=" * 72)

assert MANIFEST_PATH.exists(), f"Window manifest not found: {MANIFEST_PATH}"
assert TRIAL_SPLIT_MANIFEST_PATH.exists(), f"Trial split manifest not found: {TRIAL_SPLIT_MANIFEST_PATH}"

trial_split_df = pd.read_csv(TRIAL_SPLIT_MANIFEST_PATH)
authoritative_splits = {
    part: set(trial_split_df[trial_split_df['split'] == part]['trial_key'])
    for part in ['TRAIN', 'VALIDATION', 'TEST']
}
assert authoritative_splits['TRAIN'].isdisjoint(authoritative_splits['VALIDATION']), (
    "Authoritative trial_split_manifest has TRAIN/VALIDATION trial overlap!"
)
assert authoritative_splits['TRAIN'].isdisjoint(authoritative_splits['TEST']), (
    "Authoritative trial_split_manifest has TRAIN/TEST trial overlap!"
)
assert authoritative_splits['VALIDATION'].isdisjoint(authoritative_splits['TEST']), (
    "Authoritative trial_split_manifest has VALIDATION/TEST trial overlap!"
)

missing_files = []
shape_errors = []

for branch in EXPECTED_BRANCHES:
    for fold_id in EXPECTED_FOLDS:
        fm = manifest[(manifest['fold_id'] == fold_id) & (manifest['branch'] == branch)]
        if len(fm) == 0:
            raise RuntimeError(f"Manifest has no rows for branch='{branch}', fold_id='{fold_id}'")

        # 1. Check each required partition is non-empty and matches trial_split_manifest
        part_trial_sets = {}
        for part in ['TRAIN', 'VALIDATION', 'TEST']:
            pm = fm[fm['partition'] == part]
            if len(pm) == 0:
                raise RuntimeError(
                    f"Empty partition '{part}' for branch='{branch}', fold_id='{fold_id}'"
                )
            part_trial_sets[part] = set(pm['trial_key'])

        assert part_trial_sets['TRAIN'].isdisjoint(part_trial_sets['VALIDATION']), (
            f"[{branch}/{fold_id}] TRAIN and VALIDATION trial keys overlap!"
        )
        assert part_trial_sets['TRAIN'].isdisjoint(part_trial_sets['TEST']), (
            f"[{branch}/{fold_id}] TRAIN and TEST trial keys overlap!"
        )
        assert part_trial_sets['VALIDATION'].isdisjoint(part_trial_sets['TEST']), (
            f"[{branch}/{fold_id}] VALIDATION and TEST trial keys overlap!"
        )

        for part in ['TRAIN', 'VALIDATION', 'TEST']:
            if part_trial_sets[part] != authoritative_splits[part]:
                diff_a = part_trial_sets[part] ^ authoritative_splits[part]
                raise AssertionError(
                    f"[{branch}/{fold_id}] Partition '{part}' trial keys do not match "
                    f"trial_split_manifest.csv (symmetric diff count: {len(diff_a)})"
                )

        # 2. Validate window_length column in manifest
        win_lengths = set(fm['window_length'].unique())
        if win_lengths != {W}:
            raise ValueError(
                f"[{branch}/{fold_id}] Expected window_length == {W}, found {win_lengths}"
            )

        # 3. Verify all scaled feature files exist (never silently skip missing files)
        checked_sample = False
        for _, row in fm.iterrows():
            fpath = (
                PROJECT_ROOT / 'data' / 'processed' / 'scaled_features'
                / branch / str(fold_id) / f"{row['trial_key']}.csv"
            )
            if not fpath.exists():
                missing_files.append(str(fpath))
            elif not checked_sample:
                # Validate sequence and feature dimensions on disk
                sample_df = pd.read_csv(fpath)
                feat_cols = [c for c in sample_df.columns if c not in ['frame', 'time']]
                if len(sample_df) != W or len(feat_cols) != FEAT_COUNT:
                    shape_errors.append(
                        f"{fpath}: shape ({len(sample_df)}, {len(feat_cols)}) != ({W}, {FEAT_COUNT})"
                    )
                checked_sample = True

if missing_files:
    raise FileNotFoundError(
        f"Validation failed: {len(missing_files)} scaled feature files are missing!\n"
        + "\n".join(missing_files[:20])
    )
if shape_errors:
    raise ValueError(
        f"Validation failed: feature/sequence dimension mismatch!\n"
        + "\n".join(shape_errors[:10])
    )

# 4. Validate target labels
if manifest[TARGET_COL].isna().any():
    raise ValueError(f"Manifest column '{TARGET_COL}' contains missing (NaN) labels!")
observed_labels = sorted(manifest[TARGET_COL].astype(int).unique().tolist())
if observed_labels != list(range(NUM_CLASSES)):
    raise ValueError(
        f"Expected target classes {list(range(NUM_CLASSES))}, found {observed_labels}"
    )

# 5. Report partition contents and trial-wise subject overlap accurately
fm0 = manifest[(manifest['fold_id'] == 'trial_split') & (manifest['branch'] == EXPECTED_BRANCHES[0])]
tr_subs = set(fm0[fm0['partition'] == 'TRAIN']['subject_id'])
va_subs = set(fm0[fm0['partition'] == 'VALIDATION']['subject_id'])
te_subs = set(fm0[fm0['partition'] == 'TEST']['subject_id'])
shared_all = tr_subs & va_subs & te_subs
shared_any = (tr_subs & va_subs) | (tr_subs & te_subs) | (va_subs & te_subs)

print(f"All {len(manifest)} manifest entries verified across {len(EXPECTED_BRANCHES)} branches.")
print(f"Missing files             : 0")
print(f"Sequence & feature shape  : ({W}, {FEAT_COUNT}) verified (excluding frame/time)")
print(f"Target classes verified   : {observed_labels}")
print("-" * 72)
for part in ['TRAIN', 'VALIDATION', 'TEST']:
    sub_df = fm0[fm0['partition'] == part]
    cls_dist = sub_df[TARGET_COL].astype(int).value_counts().sort_index().to_dict()
    print(
        f"Partition {part:<10s} : {len(sub_df):3d} trials | "
        f"{sub_df['subject_id'].nunique():2d} subjects | class dist: {cls_dist}"
    )
print("-" * 72)
print(
    f"Note on Split Methodology : Trial-wise 70/15/15 split (`trial_split`).\n"
    f"  - Trial keys are strictly pairwise disjoint across TRAIN, VALIDATION, and TEST.\n"
    f"  - {len(shared_any)} unique subjects appear in >1 partition "
    f"({len(shared_all)} appear in all 3 partitions).\n"
    f"  - This is a trial-wise split evaluation (NOT subject-independent)."
)
print("=" * 72)
print("[PASS] All data, manifest, label, and dimension checks succeeded.")


DATA AND MANIFEST VALIDATION
All 3524 manifest entries verified across 4 branches.
Missing files             : 0
Sequence & feature shape  : (751, 180) verified (excluding frame/time)
Target classes verified   : [0, 1, 2]
------------------------------------------------------------------------
Partition TRAIN      : 619 trials | 26 subjects | class dist: {0: 269, 1: 207, 2: 143}
Partition VALIDATION : 131 trials | 26 subjects | class dist: {0: 56, 1: 44, 2: 31}
Partition TEST       : 131 trials | 26 subjects | class dist: {0: 56, 1: 44, 2: 31}
------------------------------------------------------------------------
Note on Split Methodology : Trial-wise 70/15/15 split (`trial_split`).
  - Trial keys are strictly pairwise disjoint across TRAIN, VALIDATION, and TEST.
  - 26 unique subjects appear in >1 partition (26 appear in all 3 partitions).
  - This is a trial-wise split evaluation (NOT subject-independent).
[PASS] All data, manifest, label, and dimension checks succeeded.


### Cell 3: Shared Data-Loading, Preprocessing, Checkpointing, and Training Utilities

In [3]:
# ============================================================================
# CELL 3: SHARED DATA-LOADING, PREPROCESSING, AND TRAINING FUNCTIONS
# ============================================================================

def set_seed(seed=SEED):
    """Set Python, NumPy, and PyTorch random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_rng_states():
    """Capture current RNG states for checkpoint resumption."""
    states = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        states['torch_cuda'] = torch.cuda.get_rng_state_all()
    return states


def set_rng_states(states):
    """Restore RNG states from a checkpoint dictionary."""
    try:
        if 'python' in states:
            random.setstate(states['python'])
        if 'numpy' in states:
            np.random.set_state(states['numpy'])
        if 'torch_cpu' in states:
            st = states['torch_cpu']
            if hasattr(st, 'cpu'):
                st = st.cpu()
            if hasattr(st, 'byte'):
                st = st.byte()
            torch.set_rng_state(st)
        if 'torch_cuda' in states and torch.cuda.is_available():
            sts = states['torch_cuda']
            if isinstance(sts, list):
                new_sts = []
                for s in sts:
                    if hasattr(s, 'cpu'):
                        s = s.cpu()
                    if hasattr(s, 'byte'):
                        s = s.byte()
                    new_sts.append(s)
                torch.cuda.set_rng_state_all(new_sts)
            else:
                if hasattr(sts, 'cpu'):
                    sts = sts.cpu()
                if hasattr(sts, 'byte'):
                    sts = sts.byte()
                torch.cuda.set_rng_state(sts)
    except Exception as e:
        print(f"Warning: Could not fully restore RNG states: {e}")


def get_model_config(model_name):
    """Return authoritative training configuration for each baseline model."""
    if model_name == 'BiLSTM_CNN':
        return {
            'optimizer': 'AdamW',
            'lr': 1e-4,
            'weight_decay': 0.0,
            'betas': (0.9, 0.999),
            'batch_size': 192,
            'max_epochs': BILSTM_CNN_MAX_EPOCHS,      # 400
            'patience': BILSTM_CNN_PATIENCE,          # 10
            'scheduler': 'ReduceLROnPlateau',
            'scheduler_factor': 0.5,
            'scheduler_patience': 5,
            'clip_norm': 1.0,
            'use_augmentation': False,
            'checkpoint_criterion': 'val_acc',        # Val accuracy primary, val loss tiebreak
        }
    elif model_name == 'Classical_LR':
        return {
            'optimizer': 'LBFGS',
            'lr': 'N/A',
            'weight_decay': 'N/A',
            'batch_size': 'N/A',
            'max_epochs': 300,
            'patience': 'N/A',
            'scheduler': None,
            'clip_norm': None,
            'use_augmentation': False,
            'checkpoint_criterion': 'N/A',
        }
    elif model_name in ('CNN1D', 'LSTM', 'BiLSTM'):
        return {
            'optimizer': 'Adam',
            'lr': 1e-3,
            'weight_decay': 0.0,
            'betas': (0.9, 0.999),
            'batch_size': 32,
            'max_epochs': MAX_EPOCHS_DEFAULT,         # 100
            'patience': EARLY_STOPPING_PATIENCE,      # 10
            'scheduler': 'ReduceLROnPlateau',
            'scheduler_factor': 0.5,
            'scheduler_patience': 5,
            'clip_norm': 1.0,
            'use_augmentation': False,
            'checkpoint_criterion': 'val_loss',       # Val loss primary
        }
    else:
        raise ValueError(f"Unknown model_name: {model_name}")


def apply_augmentations(bx):
    """Optional Gaussian noise + temporal shift augmentation."""
    noise = torch.randn_like(bx) * 0.01
    bx = bx + noise
    shift = np.random.randint(-5, 6)
    if shift > 0:
        bx = TF.pad(bx[:, :-shift, :], pad=(0, 0, shift, 0), mode='replicate')
    elif shift < 0:
        bx = TF.pad(bx[:, -shift:, :], pad=(0, 0, 0, -shift), mode='replicate')
    return bx


# In-memory cache of loaded & imputed branch data so independent cells run fast
_DATA_CACHE = {}

def load_and_impute_fold_data(branch, fold_id='trial_split', use_cache=True):
    """
    Load pre-scaled sequence CSVs for (branch, fold_id).
    Fits median SimpleImputer strictly on TRAIN and applies to TRAIN, VALIDATION, TEST.
    """
    cache_key = (branch, fold_id)
    if use_cache and cache_key in _DATA_CACHE:
        return copy.deepcopy(_DATA_CACHE[cache_key])

    fm = manifest[(manifest['fold_id'] == fold_id) & (manifest['branch'] == branch)]
    if len(fm) == 0:
        raise RuntimeError(f"No manifest rows found for branch='{branch}', fold_id='{fold_id}'")

    tr_trials = set(fm[fm['partition'] == 'TRAIN']['trial_key'])
    va_trials = set(fm[fm['partition'] == 'VALIDATION']['trial_key'])
    te_trials = set(fm[fm['partition'] == 'TEST']['trial_key'])

    assert len(tr_trials) > 0 and len(va_trials) > 0 and len(te_trials) > 0, (
        f"Empty partition detected in {branch}/{fold_id}!"
    )
    assert tr_trials.isdisjoint(va_trials), "TRAIN and VALIDATION trials overlap!"
    assert tr_trials.isdisjoint(te_trials), "TRAIN and TEST trials overlap!"
    assert va_trials.isdisjoint(te_trials), "VALIDATION and TEST trials overlap!"

    raw_splits = {}
    for part, canon_part in [('TRAIN', 'train'), ('VALIDATION', 'val'), ('TEST', 'test')]:
        pm = fm[fm['partition'] == part]
        Xs, ys, metas = [], [], []
        for _, row in pm.iterrows():
            fpath = (
                PROJECT_ROOT / 'data' / 'processed' / 'scaled_features'
                / branch / str(fold_id) / f"{row['trial_key']}.csv"
            )
            if not fpath.exists():
                raise FileNotFoundError(
                    f"Required scaled feature file missing: {fpath}"
                )
            df = pd.read_csv(fpath)
            feat_cols = [c for c in df.columns if c not in ['frame', 'time']]
            if len(feat_cols) != FEAT_COUNT:
                raise ValueError(
                    f"{fpath}: expected {FEAT_COUNT} features (excluding frame/time), got {len(feat_cols)}"
                )
            arr = df[feat_cols].to_numpy(dtype=np.float32)
            if arr.shape != (W, FEAT_COUNT):
                raise ValueError(
                    f"{fpath}: expected sequence shape ({W}, {FEAT_COUNT}), got {arr.shape}"
                )
            Xs.append(arr)
            ys.append(int(row[TARGET_COL]))
            metas.append({
                'subject_id': str(row['subject_id']),
                'trial_key': str(row['trial_key']),
                'condition': str(row.get('condition', '')),
            })

        if len(Xs) == 0:
            raise RuntimeError(f"Loaded 0 trials for {branch}/{fold_id}/{part}!")

        raw_splits[canon_part] = {
            'X': np.stack(Xs, axis=0).astype(np.float32),
            'y': np.array(ys, dtype=np.int64),
            'meta': metas,
        }

    # Fit median imputer on TRAIN only, then transform TRAIN, VAL, TEST
    X_train = raw_splits['train']['X']
    N_tr, W_tr, F_tr = X_train.shape
    assert (W_tr, F_tr) == (W, FEAT_COUNT), f"Unexpected TRAIN dims: {(W_tr, F_tr)}"

    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train.reshape(-1, F_tr))

    imp_dir = PROJECT_ROOT / 'data' / 'processed' / 'model_input_imputers' / branch / str(fold_id)
    imp_dir.mkdir(parents=True, exist_ok=True)
    with open(imp_dir / "imputer.pkl", 'wb') as f_imp:
        pickle.dump(imputer, f_imp)

    imputed_splits = {}
    for canon_part in ['train', 'val', 'test']:
        X_raw = raw_splits[canon_part]['X']
        N_p, W_p, F_p = X_raw.shape
        X_imp = imputer.transform(X_raw.reshape(-1, F_p)).reshape(N_p, W_p, F_p).astype(np.float32)
        if not np.isfinite(X_imp).all():
            raise ValueError(f"Non-finite values remain after imputation in {branch}/{canon_part}!")
        imputed_splits[canon_part] = {
            'X': X_imp,
            'y': raw_splits[canon_part]['y'],
            'meta': raw_splits[canon_part]['meta'],
        }

    if use_cache:
        _DATA_CACHE[cache_key] = copy.deepcopy(imputed_splits)
    return imputed_splits


def calc_class_weights(y_train):
    """Compute inverse-frequency class weights from TRAIN labels only."""
    counts = np.bincount(y_train.astype(int), minlength=NUM_CLASSES)
    if np.any(counts == 0):
        raise ValueError(f"Zero-count class in TRAIN labels: {counts}")
    weights = np.sum(counts) / (float(NUM_CLASSES) * counts.astype(float))
    return torch.tensor(weights, dtype=torch.float32)


def create_dataloaders(splits, batch_size):
    """Create PyTorch DataLoaders for train, val, and test partitions."""
    train_ds = TensorDataset(
        torch.tensor(splits['train']['X'], dtype=torch.float32),
        torch.tensor(splits['train']['y'], dtype=torch.long),
    )
    val_ds = TensorDataset(
        torch.tensor(splits['val']['X'], dtype=torch.float32),
        torch.tensor(splits['val']['y'], dtype=torch.long),
    )
    test_ds = TensorDataset(
        torch.tensor(splits['test']['X'], dtype=torch.float32),
        torch.tensor(splits['test']['y'], dtype=torch.long),
    )
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_ld   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_ld  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    return train_ld, val_ld, test_ld


def compute_eval_metrics(y_true, y_pred):
    """Compute comprehensive evaluation metrics, confusion matrix, and report."""
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    base = compute_classification_metrics(y_true, y_pred, num_classes=NUM_CLASSES)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'cohen_kappa': float(cohen_kappa_score(y_true, y_pred)),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES))),
        'classification_report': classification_report(
            y_true, y_pred, labels=list(range(NUM_CLASSES)),
            target_names=[f"Class {i}" for i in range(NUM_CLASSES)],
             digits=4, zero_division=0
        ),
        'detailed': base,
    }


def safe_replace(src, dst):
    """Atomically replace dst with src, retrying on transient Windows file locks."""
    for _ in range(10):
        try:
            os.replace(src, dst)
            return
        except PermissionError:
            time.sleep(0.1)
    if os.path.exists(dst):
        os.remove(dst)
    os.replace(src, dst)


def make_run_id():
    """Create a timestamped unique run identifier."""
    return datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]


def get_checkpoint_paths(model_name, branch, fold_id, run_id=None, create_new_run_dir=False):
    """
    Resolve checkpoint paths.
    When create_new_run_dir=True (used for fresh runs so old results are never overwritten),
    creates a versioned subdirectory `run_<run_id>` under `fold_<fold_id>`.
    """
    fold_base = PROJECT_ROOT / 'results' / 'checkpoints' / model_name / branch / f"fold_{fold_id}"
    fold_base.mkdir(parents=True, exist_ok=True)
    if create_new_run_dir:
        assert run_id is not None, "run_id required when create_new_run_dir=True"
        base = fold_base / f"run_{run_id}"
        base.mkdir(parents=True, exist_ok=True)
    else:
        base = fold_base
    ext = ".pkl" if model_name == "Classical_LR" else ".pt"
    return (
        base / f"best_checkpoint{ext}",
        base / f"last_checkpoint{ext}",
        base / "training_history.csv",
        base / "checkpoint_metadata.json",
        fold_base / "latest_run.json",
    )


def validate_checkpoint_metadata(ckpt, meta_path, expected_meta):
    """
    Validate that a saved checkpoint matches the expected model, branch, split,
    feature count (180), sequence length (751), training config, checkpoint criterion, and run_id.
    Rejects legacy or incompatible checkpoints instead of loading them silently.
    Returns (is_valid: bool, reason: str).
    """
    meta = {}
    if meta_path is not None and Path(meta_path).exists():
        try:
            with open(meta_path, 'r') as f:
                meta = json.load(f)
        except Exception as e:
            return False, f"Failed to read metadata JSON: {e}"

    if isinstance(ckpt, dict):
        for k in ['model', 'branch', 'fold', 'split', 'feature_count', 'seq_length',
                  'checkpoint_criterion', 'config', 'run_id']:
            if k in ckpt and k not in meta:
                meta[k] = ckpt[k]

    required_keys = ['model', 'branch', 'fold', 'feature_count', 'seq_length',
                     'checkpoint_criterion', 'config', 'run_id']
    for rk in required_keys:
        if rk not in meta or meta[rk] is None:
            return False, f"Missing required metadata field '{rk}'"

    if meta.get('model') != expected_meta['model']:
        return False, f"Model mismatch: '{meta.get('model')}' != '{expected_meta['model']}'"
    if meta.get('branch') != expected_meta['branch']:
        return False, f"Branch mismatch: '{meta.get('branch')}' != '{expected_meta['branch']}'"
    if meta.get('fold') != expected_meta['fold']:
        return False, f"Split/fold mismatch: '{meta.get('fold')}' != '{expected_meta['fold']}'"
    if int(meta.get('feature_count', -1)) != int(expected_meta['feature_count']):
        return False, f"Feature count mismatch: {meta.get('feature_count')} != {expected_meta['feature_count']}"
    if int(meta.get('seq_length', -1)) != int(expected_meta['seq_length']):
        return False, f"Sequence length mismatch: {meta.get('seq_length')} != {expected_meta['seq_length']}"
    if meta.get('checkpoint_criterion') != expected_meta['checkpoint_criterion']:
        return (
            False,
            f"Checkpoint criterion mismatch: '{meta.get('checkpoint_criterion')}' != '{expected_meta['checkpoint_criterion']}'"
        )

    exp_cfg_sub = expected_meta.get('config', {})
    act_cfg_sub = meta.get('config', {})
    for ck in ['optimizer', 'lr', 'batch_size']:
        if ck in exp_cfg_sub and act_cfg_sub.get(ck) != exp_cfg_sub.get(ck):
            return False, f"Config mismatch on '{ck}': {act_cfg_sub.get(ck)} != {exp_cfg_sub.get(ck)}"

    return True, "OK"


def save_model_results(model_name, results_list, history_list, predictions_list):
    """
    Persist per-model CSV files AND safely update consolidated CSV files
    without overwriting other models' results.
    """
    SESSION_MODEL_RESULTS[model_name]     = copy.deepcopy(results_list)
    SESSION_MODEL_HISTORY[model_name]     = copy.deepcopy(history_list)
    SESSION_MODEL_PREDICTIONS[model_name] = copy.deepcopy(predictions_list)

    tables_dir = PROJECT_ROOT / 'results' / 'tables'
    preds_dir  = PROJECT_ROOT / 'results' / 'predictions'

    # 1. Per-model files
    res_df = pd.DataFrame(results_list)
    res_df.to_csv(tables_dir / f"{model_name}_test_metrics.csv", index=False)

    if history_list:
        hist_df = pd.DataFrame(history_list)
        hist_df.to_csv(tables_dir / f"{model_name}_training_history.csv", index=False)

    if predictions_list:
        pred_df = pd.DataFrame(predictions_list)
        pred_df.to_csv(preds_dir / f"{model_name}_test_predictions.csv", index=False)

    # 2. Safely merge into consolidated files (preserve other models' rows)
    cons_metrics_path = tables_dir / 'baseline_final_test_metrics.csv'
    if cons_metrics_path.exists():
        try:
            existing_m = pd.read_csv(cons_metrics_path)
            existing_m = existing_m[existing_m['model'] != model_name]
            merged_m = pd.concat([existing_m, res_df], ignore_index=True)
        except Exception:
            merged_m = res_df
    else:
        merged_m = res_df
    merged_m.to_csv(cons_metrics_path, index=False)

    if history_list:
        cons_hist_path = tables_dir / 'baseline_training_history.csv'
        if cons_hist_path.exists():
            try:
                existing_h = pd.read_csv(cons_hist_path)
                existing_h = existing_h[existing_h['model'] != model_name]
                merged_h = pd.concat([existing_h, hist_df], ignore_index=True)
            except Exception:
                merged_h = hist_df
        else:
            merged_h = hist_df
        merged_h.to_csv(cons_hist_path, index=False)

    if predictions_list:
        cons_pred_path = preds_dir / 'baseline_test_predictions.csv'
        if cons_pred_path.exists():
            try:
                existing_p = pd.read_csv(cons_pred_path)
                existing_p = existing_p[existing_p['model'] != model_name]
                merged_p = pd.concat([existing_p, pred_df], ignore_index=True)
            except Exception:
                merged_p = pred_df
        else:
            merged_p = pred_df
        merged_p.to_csv(cons_pred_path, index=False)


def train_deep_model_for_branches(
    model_name,
    model_factory,
    branches=EXPECTED_BRANCHES,
    fold_ids=EXPECTED_FOLDS,
    force_retrain=False,
    max_epochs_override=None,
):
    """
    Train and evaluate a deep learning model across all specified branches.
    - Initializes a fresh model, optimizer, and scheduler for each branch.
    - Supports both 'val_loss' selection (CNN1D, LSTM, BiLSTM) and 'val_acc' selection
      with val_loss tie-breaking and strict val_acc early stopping (BiLSTM_CNN).
    - When force_retrain=True, writes to a new versioned run directory so previous
      checkpoints are preserved and training is guaranteed to execute from epoch 1.
    """
    cfg = copy.deepcopy(get_model_config(model_name))
    if max_epochs_override is not None:
        cfg['max_epochs'] = int(max_epochs_override)

    max_epochs           = int(cfg['max_epochs'])
    patience             = int(cfg['patience'])
    checkpoint_criterion = cfg['checkpoint_criterion']
    run_id               = make_run_id()

    model_results     = []
    model_history     = []
    model_predictions = []

    print("\n" + "=" * 88)
    print(f"STARTING DEEP MODEL TRAINING: {model_name} | Run ID: {run_id}")
    print("=" * 88)

    for branch in branches:
        for fold_id in fold_ids:
            set_seed(SEED)
            splits = load_and_impute_fold_data(branch, fold_id)
            n_train = int(splits['train']['X'].shape[0])
            n_val   = int(splits['val']['X'].shape[0])
            n_test  = int(splits['test']['X'].shape[0])

            expected_meta = {
                'model': model_name,
                'branch': branch,
                'fold': fold_id,
                'feature_count': FEAT_COUNT,
                'seq_length': W,
                'checkpoint_criterion': checkpoint_criterion,
                'config': cfg,
            }

            best_ckpt_path, last_ckpt_path, hist_path, meta_path, latest_ptr_path = get_checkpoint_paths(
                model_name, branch, fold_id, run_id=run_id, create_new_run_dir=force_retrain
            )

            # Initialize fresh model, loss criteria, optimizer, and scheduler
            train_ld, val_ld, test_ld = create_dataloaders(splits, cfg['batch_size'])
            cw_tensor = calc_class_weights(splits['train']['y']).to(device)

            model = model_factory().to(device)
            train_criterion = nn.CrossEntropyLoss(weight=cw_tensor)
            val_criterion   = nn.CrossEntropyLoss()

            if cfg['optimizer'] == 'AdamW':
                optimizer = torch.optim.AdamW(
                    model.parameters(),
                    lr=cfg['lr'],
                    betas=cfg['betas'],
                    weight_decay=cfg.get('weight_decay', 0.0),
                )
            else:
                optimizer = torch.optim.Adam(
                    model.parameters(),
                    lr=cfg['lr'],
                    betas=cfg['betas'],
                    weight_decay=cfg.get('weight_decay', 0.0),
                )

            scheduler = (
                torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer,
                    mode='min',
                    factor=cfg['scheduler_factor'],
                    patience=cfg['scheduler_patience'],
                )
                if cfg['scheduler'] == 'ReduceLROnPlateau'
                else None
            )

            start_epoch      = 1
            best_val_loss    = float('inf')
            best_val_acc     = -1.0
            best_val_bacc    = 0.0
            best_val_f1      = 0.0
            best_epoch       = 0
            patience_counter = 0
            history          = []
            run_mode         = "FRESH"

            # Check existing checkpoint only if force_retrain is False
            if not force_retrain and last_ckpt_path.exists():
                try:
                    ckpt = torch.load(last_ckpt_path, map_location=device, weights_only=False)
                    is_valid, reason = validate_checkpoint_metadata(ckpt, meta_path, expected_meta)
                    if not is_valid:
                        print(f"  [!] Existing checkpoint rejected ({reason}). Starting FRESH run in new versioned directory.")
                        best_ckpt_path, last_ckpt_path, hist_path, meta_path, latest_ptr_path = get_checkpoint_paths(
                            model_name, branch, fold_id, run_id=run_id, create_new_run_dir=True
                        )
                        run_mode = "FRESH"
                    else:
                        ckpt_epoch = int(ckpt['epoch'])
                        ckpt_patience = int(ckpt.get('patience_counter', 0))
                        best_val_loss = float(ckpt['best_val_loss'])
                        best_val_acc  = float(ckpt.get('best_val_accuracy', -1.0))
                        best_val_bacc = float(ckpt.get('best_val_balanced_accuracy', 0.0))
                        best_val_f1   = float(ckpt.get('best_val_macro_f1', 0.0))
                        best_epoch    = int(ckpt['best_epoch'])
                        history       = ckpt.get('history', [])

                        if ckpt_epoch >= max_epochs or ckpt_patience >= patience:
                            run_mode = "REUSED_COMPLETED"
                            start_epoch = max_epochs + 1
                        else:
                            run_mode = "RESUMED"
                            model.load_state_dict(ckpt['model_state_dict'])
                            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                            if scheduler and 'scheduler_state_dict' in ckpt:
                                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
                            if 'rng_states' in ckpt:
                                set_rng_states(ckpt['rng_states'])
                            start_epoch = ckpt_epoch + 1
                            patience_counter = ckpt_patience
                except Exception as e:
                    print(f"  [!] Failed to load existing checkpoint ({e}). Starting FRESH run in new versioned directory.")
                    best_ckpt_path, last_ckpt_path, hist_path, meta_path, latest_ptr_path = get_checkpoint_paths(
                        model_name, branch, fold_id, run_id=run_id, create_new_run_dir=True
                    )
                    run_mode = "FRESH"

            crit_desc = (
                "Validation Accuracy (primary) + Validation Loss (tie-breaker)"
                if checkpoint_criterion == 'val_acc'
                else "Validation Loss (primary)"
            )
            print("\n" + "-" * 88)
            print(f"Model                 : {model_name}")
            print(f"Branch                : {branch}")
            print(f"Split / Fold          : {fold_id}")
            print(f"Partition Sizes       : TRAIN={n_train} | VALIDATION={n_val} | TEST={n_test}")
            print(f"Device                : {device}")
            print(f"Optimizer             : {cfg['optimizer']} (lr={cfg['lr']}, weight_decay={cfg.get('weight_decay', 0.0)})")
            print(f"Batch Size            : {cfg['batch_size']}")
            print(f"Max Epochs / Patience : {max_epochs} / {patience}")
            print(f"Augmentation          : {cfg['use_augmentation']}")
            print(f"Scheduler             : {cfg['scheduler']} (monitors val_loss, factor={cfg['scheduler_factor']})")
            print(f"Checkpoint Criterion  : {crit_desc}")
            print(f"Run Mode              : {run_mode} (force_retrain={force_retrain})")
            print(f"Checkpoint Directory  : {best_ckpt_path.parent}")
            print("-" * 88)

            if run_mode == "REUSED_COMPLETED":
                print(
                    f"  [REUSED RESULT] Completed checkpoint found (stopped at epoch {len(history)}, "
                    f"best epoch {best_epoch}). Reusing saved checkpoint â€” NOT a new training run."
                )

            epochs_executed_this_run = 0

            for epoch in range(start_epoch, max_epochs + 1):
                ep_t0 = time.time()

                # â”€â”€ TRAIN â”€â”€
                model.train()
                tr_loss_sum, tr_preds, tr_trues = 0.0, [], []
                for bx, by in train_ld:
                    bx, by = bx.to(device), by.to(device)
                    if cfg['use_augmentation']:
                        bx = apply_augmentations(bx)
                    optimizer.zero_grad()
                    out = model(bx)
                    loss = train_criterion(out, by)
                    loss.backward()
                    if cfg['clip_norm']:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip_norm'])
                    optimizer.step()
                    tr_loss_sum += loss.item() * len(by)
                    tr_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                    tr_trues.extend(by.cpu().numpy())

                tr_loss = tr_loss_sum / len(train_ld.dataset)
                tr_acc  = float(accuracy_score(tr_trues, tr_preds))
                tr_bacc = float(balanced_accuracy_score(tr_trues, tr_preds))
                tr_f1   = float(f1_score(tr_trues, tr_preds, average='macro', zero_division=0))

                # â”€â”€ VALIDATION â”€â”€
                model.eval()
                vl_loss_sum, vl_preds, vl_trues = 0.0, [], []
                with torch.no_grad():
                    for bx, by in val_ld:
                        bx, by = bx.to(device), by.to(device)
                        out = model(bx)
                        loss = val_criterion(out, by)
                        vl_loss_sum += loss.item() * len(by)
                        vl_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                        vl_trues.extend(by.cpu().numpy())

                vl_loss = vl_loss_sum / len(val_ld.dataset)
                vl_acc  = float(accuracy_score(vl_trues, vl_preds))
                vl_bacc = float(balanced_accuracy_score(vl_trues, vl_preds))
                vl_f1   = float(f1_score(vl_trues, vl_preds, average='macro', zero_division=0))

                current_lr = float(optimizer.param_groups[0]['lr'])
                ep_time = time.time() - ep_t0

                if scheduler is not None:
                    scheduler.step(vl_loss)

                # â”€â”€ CHECKPOINT SELECTION & EARLY STOPPING â”€â”€
                if checkpoint_criterion == 'val_acc':
                    # BiLSTM-CNN rules:
                    # 1. Save new best checkpoint whenever val_acc strictly improves.
                    # 2. When val_acc ties best_val_acc, replace checkpoint only if val_loss is lower.
                    # 3. Reset patience_counter ONLY when val_acc strictly improves.
                    # 4. Lower val_loss alone never resets patience_counter.
                    strictly_improved = vl_acc > best_val_acc
                    tied_with_lower_loss = (
                        np.isclose(vl_acc, best_val_acc, rtol=0.0, atol=1e-12)
                        and (vl_loss < best_val_loss)
                    )
                    is_best = bool(strictly_improved or tied_with_lower_loss)
                    if strictly_improved:
                        patience_counter = 0
                    else:
                        patience_counter += 1
                else:
                    # Default val_loss criterion for other deep baselines
                    is_best = bool(vl_loss < best_val_loss)
                    if is_best:
                        patience_counter = 0
                    else:
                        patience_counter += 1

                if is_best:
                    best_val_loss = vl_loss
                    best_val_acc  = vl_acc
                    best_val_bacc = vl_bacc
                    best_val_f1   = vl_f1
                    best_epoch    = epoch

                h_rec = {
                    'model': model_name,
                    'branch': branch,
                    'fold': fold_id,
                    'run_id': run_id,
                    'epoch': epoch,
                    'train_loss': tr_loss,
                    'val_loss': vl_loss,
                    'train_accuracy': tr_acc,
                    'val_accuracy': vl_acc,
                    'train_balanced_accuracy': tr_bacc,
                    'val_balanced_accuracy': vl_bacc,
                    'train_macro_f1': tr_f1,
                    'val_macro_f1': vl_f1,
                    'learning_rate': current_lr,
                    'epoch_time': ep_time,
                    'is_best': is_best,
                }
                history.append(h_rec)
                epochs_executed_this_run += 1

                best_marker = " *BEST*" if is_best else ""
                print(
                    f"Epoch {epoch:03d}/{max_epochs:03d} | "
                    f"TrLoss: {tr_loss:.4f} | VLoss: {vl_loss:.4f} | "
                    f"TrAcc: {tr_acc:.4f} | VAcc: {vl_acc:.4f} | "
                    f"LR: {current_lr:.2e} | Pat: {patience_counter:02d}/{patience:02d} | "
                    f"{ep_time:.1f}s{best_marker}"
                )

                ckpt_payload = {
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'epoch': epoch,
                    'best_epoch': best_epoch,
                    'best_val_loss': best_val_loss,
                    'best_val_accuracy': best_val_acc,
                    'best_val_balanced_accuracy': best_val_bacc,
                    'best_val_macro_f1': best_val_f1,
                    'patience_counter': patience_counter,
                    'history': history,
                    'rng_states': get_rng_states(),
                    'model': model_name,
                    'branch': branch,
                    'fold': fold_id,
                    'run_id': run_id,
                    'feature_count': FEAT_COUNT,
                    'seq_length': W,
                    'checkpoint_criterion': checkpoint_criterion,
                    'config': cfg,
                }
                if scheduler is not None:
                    ckpt_payload['scheduler_state_dict'] = scheduler.state_dict()

                if is_best:
                    torch.save(ckpt_payload, str(best_ckpt_path) + ".tmp")
                    safe_replace(str(best_ckpt_path) + ".tmp", str(best_ckpt_path))

                torch.save(ckpt_payload, str(last_ckpt_path) + ".tmp")
                safe_replace(str(last_ckpt_path) + ".tmp", str(last_ckpt_path))

                if patience_counter >= patience:
                    crit_label = "Validation accuracy" if checkpoint_criterion == 'val_acc' else "Validation loss"
                    print(
                        f"Early stopping triggered at epoch {epoch}: "
                        f"{crit_label} did not strictly improve for {patience} consecutive epochs."
                    )
                    break

            # Save per-branch training history and metadata if training ran
            if epochs_executed_this_run > 0:
                pd.DataFrame(history).to_csv(hist_path, index=False)
                meta_record = {
                    'model': model_name,
                    'branch': branch,
                    'fold': fold_id,
                    'run_id': run_id,
                    'run_mode': run_mode,
                    'feature_count': FEAT_COUNT,
                    'seq_length': W,
                    'checkpoint_criterion': checkpoint_criterion,
                    'epochs_executed_this_run': epochs_executed_this_run,
                    'total_epochs_recorded': len(history),
                    'best_epoch': best_epoch,
                    'best_val_accuracy': best_val_acc,
                    'best_val_loss': best_val_loss,
                    'config': cfg,
                    'best_checkpoint_path': str(best_ckpt_path),
                    'timestamp': datetime.now().isoformat(),
                }
                with open(meta_path, 'w') as mf:
                    json.dump(meta_record, mf, indent=2)
                with open(latest_ptr_path, 'w') as lf:
                    json.dump(meta_record, lf, indent=2)

            model_history.extend(history)

            # â”€â”€ RELOAD BEST CHECKPOINT BEFORE FINAL TEST EVALUATION â”€â”€
            if not best_ckpt_path.exists():
                raise RuntimeError(
                    f"Best checkpoint missing at {best_ckpt_path} for {model_name}/{branch}/{fold_id}!"
                )

            best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
            model.load_state_dict(best_ckpt['model_state_dict'])
            model.eval()

            vl_best_loss = float(best_ckpt['best_val_loss'])
            vl_best_acc  = float(best_ckpt['best_val_accuracy'])
            vl_best_bacc = float(best_ckpt.get('best_val_balanced_accuracy', 0.0))
            vl_best_f1   = float(best_ckpt.get('best_val_macro_f1', 0.0))
            best_ep      = int(best_ckpt['best_epoch'])

            te_preds, te_trues, te_probs = [], [], []
            with torch.no_grad():
                for bx, by in test_ld:
                    out = model(bx.to(device))
                    probs = TF.softmax(out, dim=1)
                    te_probs.extend(probs.cpu().numpy())
                    te_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                    te_trues.extend(by.cpu().numpy())

            ev = compute_eval_metrics(te_trues, te_preds)
            is_freshly_trained = bool(epochs_executed_this_run > 0 and best_ckpt_path.exists())

            m_rec = {
                'model': model_name,
                'branch': branch,
                'fold': fold_id,
                'run_id': best_ckpt.get('run_id', run_id),
                'run_mode': run_mode if is_freshly_trained else 'REUSED_COMPLETED',
                'is_fresh_run': is_freshly_trained,
                'checkpoint_criterion': checkpoint_criterion,
                'n_train': n_train,
                'n_val': n_val,
                'n_test': n_test,
                'epochs_executed': epochs_executed_this_run if is_freshly_trained else len(history),
                'epochs_executed_this_run': epochs_executed_this_run,
                'best_epoch': best_ep,
                'best_val_loss': vl_best_loss,
                'best_val_accuracy': vl_best_acc,
                'best_val_balanced_accuracy': vl_best_bacc,
                'best_val_macro_f1': vl_best_f1,
                'test_accuracy': ev['accuracy'],
                'test_balanced_accuracy': ev['balanced_accuracy'],
                'test_macro_f1': ev['macro_f1'],
                'test_weighted_f1': ev['weighted_f1'],
                'cohen_kappa': ev['cohen_kappa'],
                'mae': ev['mae'],
                'confusion_matrix': json.dumps(ev['confusion_matrix'].tolist()),
                'checkpoint_path': str(best_ckpt_path),
            }
            model_results.append(m_rec)

            for i in range(len(te_preds)):
                model_predictions.append({
                    'model': model_name,
                    'branch': branch,
                    'fold': fold_id,
                    'run_id': best_ckpt.get('run_id', run_id),
                    'subject_id': splits['test']['meta'][i]['subject_id'],
                    'trial_id': splits['test']['meta'][i]['trial_key'],
                    'true_label': int(te_trues[i]),
                    'pred_label': int(te_preds[i]),
                    'prob_0': float(te_probs[i][0]),
                    'prob_1': float(te_probs[i][1]),
                    'prob_2': float(te_probs[i][2]),
                })

            print("\n" + "=" * 68)
            print(f"HELD-OUT TEST EVALUATION â€” {model_name} | {branch} | {fold_id}")
            print("=" * 68)
            print(f"Run Status               : {'FRESHLY TRAINED' if is_freshly_trained else 'REUSED CHECKPOINT'}")
            print(f"Partition Samples        : TRAIN={n_train}, VAL={n_val}, TEST={n_test}")
            print(f"Epochs Executed This Run : {epochs_executed_this_run} (Total History: {len(history)})")
            print(f"Selected Best Epoch      : {best_ep}")
            print(f"Best Validation Accuracy : {vl_best_acc:.4f}")
            print(f"Best Validation Loss     : {vl_best_loss:.4f}")
            print(f"TEST Accuracy            : {ev['accuracy']:.4f}")
            print(f"TEST Balanced Accuracy   : {ev['balanced_accuracy']:.4f}")
            print(f"TEST Macro F1            : {ev['macro_f1']:.4f}")
            print(f"TEST Weighted F1         : {ev['weighted_f1']:.4f}")
            print(f"TEST Cohen's Kappa       : {ev['cohen_kappa']:.4f}")
            print(f"TEST MAE                 : {ev['mae']:.4f}")
            print("\nConfusion Matrix (rows=true, cols=pred):")
            print(ev['confusion_matrix'])
            print("\nClassification Report:")
            print(ev['classification_report'])
            print("=" * 68)

    save_model_results(model_name, model_results, model_history, model_predictions)
    return model_results, model_history, model_predictions


def train_classical_lr_for_branches(
    model_factory,
    branches=EXPECTED_BRANCHES,
    fold_ids=EXPECTED_FOLDS,
    force_retrain=False,
):
    """
    Train and evaluate Classical Logistic Regression independently for each branch.
    Uses statistical feature extraction (mean + std across 751 frames -> 360 features)
    fitted on TRAIN only, evaluates on VALIDATION and held-out TEST.
    """
    model_name        = 'Classical_LR'
    cfg               = get_model_config(model_name)
    run_id            = make_run_id()
    model_results     = []
    model_predictions = []

    print("\n" + "=" * 88)
    print(f"STARTING CLASSICAL BASELINE TRAINING: {model_name} | Run ID: {run_id}")
    print("=" * 88)

    for branch in branches:
        for fold_id in fold_ids:
            set_seed(SEED)
            splits = load_and_impute_fold_data(branch, fold_id)
            n_train = int(splits['train']['X'].shape[0])
            n_val   = int(splits['val']['X'].shape[0])
            n_test  = int(splits['test']['X'].shape[0])

            best_ckpt_path, _, _, meta_path, latest_ptr_path = get_checkpoint_paths(
                model_name, branch, fold_id, run_id=run_id, create_new_run_dir=force_retrain
            )

            X_tr_feat  = extract_statistical_features(splits['train']['X'])
            X_val_feat = extract_statistical_features(splits['val']['X'])
            X_te_feat  = extract_statistical_features(splits['test']['X'])

            is_freshly_trained = False
            run_mode = "FRESH"

            if not force_retrain and best_ckpt_path.exists():
                with open(best_ckpt_path, 'rb') as f:
                    lr_model = pickle.load(f)
                run_mode = "REUSED_COMPLETED"
                print(f"\n[{model_name} | {branch}] Reusing existing checkpoint at {best_ckpt_path} (not a fresh run).")
            else:
                lr_model = model_factory()
                lr_model.fit(X_tr_feat, splits['train']['y'])
                with open(str(best_ckpt_path) + ".tmp", 'wb') as f:
                    pickle.dump(lr_model, f)
                safe_replace(str(best_ckpt_path) + ".tmp", str(best_ckpt_path))
                is_freshly_trained = True

                meta_record = {
                    'model': model_name,
                    'branch': branch,
                    'fold': fold_id,
                    'run_id': run_id,
                    'feature_count': FEAT_COUNT,
                    'seq_length': W,
                    'stat_feature_dim': int(X_tr_feat.shape[1]),
                    'n_iter': int(np.max(lr_model.n_iter_)),
                    'timestamp': datetime.now().isoformat(),
                }
                with open(meta_path, 'w') as mf:
                    json.dump(meta_record, mf, indent=2)
                with open(latest_ptr_path, 'w') as lf:
                    json.dump(meta_record, lf, indent=2)

            n_iters = int(np.max(lr_model.n_iter_)) if hasattr(lr_model, 'n_iter_') else 1

            # Validation evaluation
            val_preds = lr_model.predict(X_val_feat)
            val_probs = lr_model.predict_proba(X_val_feat)
            val_ev    = compute_eval_metrics(splits['val']['y'], val_preds)
            val_loss  = float(log_loss(splits['val']['y'], val_probs, labels=list(range(NUM_CLASSES))))

            # Held-out TEST evaluation
            te_preds = lr_model.predict(X_te_feat)
            te_probs = lr_model.predict_proba(X_te_feat)
            te_ev    = compute_eval_metrics(splits['test']['y'], te_preds)

            m_rec = {
                'model': model_name,
                'branch': branch,
                'fold': fold_id,
                'run_id': run_id,
                'run_mode': run_mode,
                'is_fresh_run': is_freshly_trained,
                'checkpoint_criterion': 'lbfgs_convergence',
                'n_train': n_train,
                'n_val': n_val,
                'n_test': n_test,
                'epochs_executed': n_iters if is_freshly_trained else 0,
                'epochs_executed_this_run': n_iters if is_freshly_trained else 0,
                'best_epoch': n_iters,
                'best_val_loss': val_loss,
                'best_val_accuracy': val_ev['accuracy'],
                'best_val_balanced_accuracy': val_ev['balanced_accuracy'],
                'best_val_macro_f1': val_ev['macro_f1'],
                'test_accuracy': te_ev['accuracy'],
                'test_balanced_accuracy': te_ev['balanced_accuracy'],
                'test_macro_f1': te_ev['macro_f1'],
                'test_weighted_f1': te_ev['weighted_f1'],
                'cohen_kappa': te_ev['cohen_kappa'],
                'mae': te_ev['mae'],
                'confusion_matrix': json.dumps(te_ev['confusion_matrix'].tolist()),
                'checkpoint_path': str(best_ckpt_path),
            }
            model_results.append(m_rec)

            for i in range(len(te_preds)):
                model_predictions.append({
                    'model': model_name,
                    'branch': branch,
                    'fold': fold_id,
                    'run_id': run_id,
                    'subject_id': splits['test']['meta'][i]['subject_id'],
                    'trial_id': splits['test']['meta'][i]['trial_key'],
                    'true_label': int(splits['test']['y'][i]),
                    'pred_label': int(te_preds[i]),
                    'prob_0': float(te_probs[i][0]),
                    'prob_1': float(te_probs[i][1]),
                    'prob_2': float(te_probs[i][2]),
                })

            print("\n" + "=" * 68)
            print(f"HELD-OUT TEST EVALUATION â€” {model_name} | {branch} | {fold_id}")
            print("=" * 68)
            print(f"Run Status               : {'FRESHLY TRAINED' if is_freshly_trained else 'REUSED CHECKPOINT'}")
            print(f"Partition Samples        : TRAIN={n_train}, VAL={n_val}, TEST={n_test}")
            print(f"LBFGS Iterations         : {n_iters}")
            print(f"Validation Accuracy      : {val_ev['accuracy']:.4f} (Val LogLoss: {val_loss:.4f})")
            print(f"TEST Accuracy            : {te_ev['accuracy']:.4f}")
            print(f"TEST Balanced Accuracy   : {te_ev['balanced_accuracy']:.4f}")
            print(f"TEST Macro F1            : {te_ev['macro_f1']:.4f}")
            print(f"TEST Weighted F1         : {te_ev['weighted_f1']:.4f}")
            print(f"TEST Cohen's Kappa       : {te_ev['cohen_kappa']:.4f}")
            print(f"TEST MAE                 : {te_ev['mae']:.4f}")
            print("\nConfusion Matrix (rows=true, cols=pred):")
            print(te_ev['confusion_matrix'])
            print("\nClassification Report:")
            print(te_ev['classification_report'])
            print("=" * 68)

    save_model_results(model_name, model_results, [], model_predictions)
    return model_results, [], model_predictions


# Print configured hyperparameters table
print("=" * 100)
print("CONFIGURED MODEL HYPERPARAMETERS")
print("=" * 100)
for _m in EXPECTED_MODELS:
    _c = get_model_config(_m)
    print(
        f"Model: {_m:<14} | Opt: {str(_c['optimizer']):<6} | LR: {str(_c['lr']):<7} | "
        f"BS: {str(_c['batch_size']):<4} | MaxEpochs: {str(_c['max_epochs']):<4} | "
        f"Patience: {str(_c['patience']):<4} | Criterion: {_c['checkpoint_criterion']}"
    )
print("=" * 100)


CONFIGURED MODEL HYPERPARAMETERS
Model: CNN1D          | Opt: Adam   | LR: 0.001   | BS: 32   | MaxEpochs: 100  | Patience: 10   | Criterion: val_loss
Model: LSTM           | Opt: Adam   | LR: 0.001   | BS: 32   | MaxEpochs: 100  | Patience: 10   | Criterion: val_loss
Model: BiLSTM         | Opt: Adam   | LR: 0.001   | BS: 32   | MaxEpochs: 100  | Patience: 10   | Criterion: val_loss
Model: BiLSTM_CNN     | Opt: AdamW  | LR: 0.0001  | BS: 192  | MaxEpochs: 400  | Patience: 10   | Criterion: val_acc
Model: Classical_LR   | Opt: LBFGS  | LR: N/A     | BS: N/A  | MaxEpochs: 300  | Patience: N/A  | Criterion: N/A


---
## Section B â€” Separate Model-Construction Cells
### Cell 4: CNN1D Model (`build_cnn1d_model`)

In [4]:
# ============================================================================
# CELL 4: CNN1D MODEL CONSTRUCTION & SMOKE TEST
# ============================================================================
def build_cnn1d_model():
    """Return a fresh CNN1DBaseline instance (input: [B, 751, 180] -> output: [B, 3])."""
    return CNN1DBaseline(in_channels=FEAT_COUNT, num_classes=NUM_CLASSES)

_cnn_test = build_cnn1d_model().eval()
_x_dummy  = torch.randn(4, W, FEAT_COUNT)
with torch.no_grad():
    _out_dummy = _cnn_test(_x_dummy)

assert _out_dummy.shape == (4, NUM_CLASSES), f"Expected (4, {NUM_CLASSES}), got {_out_dummy.shape}"
assert torch.isfinite(_out_dummy).all(), "Non-finite logits in CNN1D smoke test!"
print(f"[PASS] CNN1DBaseline smoke test: input {tuple(_x_dummy.shape)} -> logits {tuple(_out_dummy.shape)}")
print(f"       Trainable parameters: {sum(p.numel() for p in _cnn_test.parameters() if p.requires_grad):,}")
del _cnn_test, _x_dummy, _out_dummy


[PASS] CNN1DBaseline smoke test: input (4, 751, 180) -> logits (4, 3)
       Trainable parameters: 230,595


### Cell 5: LSTM Model (`build_lstm_model`)

In [5]:
# ============================================================================
# CELL 5: LSTM MODEL CONSTRUCTION & SMOKE TEST
# ============================================================================
def build_lstm_model():
    """Return a fresh LSTMBaseline instance (input: [B, 751, 180] -> output: [B, 3])."""
    return LSTMBaseline(input_size=FEAT_COUNT, num_classes=NUM_CLASSES)

_lstm_test = build_lstm_model().eval()
_x_dummy   = torch.randn(4, W, FEAT_COUNT)
with torch.no_grad():
    _out_dummy = _lstm_test(_x_dummy)

assert _out_dummy.shape == (4, NUM_CLASSES), f"Expected (4, {NUM_CLASSES}), got {_out_dummy.shape}"
assert torch.isfinite(_out_dummy).all(), "Non-finite logits in LSTM smoke test!"
print(f"[PASS] LSTMBaseline smoke test: input {tuple(_x_dummy.shape)} -> logits {tuple(_out_dummy.shape)}")
print(f"       Trainable parameters: {sum(p.numel() for p in _lstm_test.parameters() if p.requires_grad):,}")
del _lstm_test, _x_dummy, _out_dummy


[PASS] LSTMBaseline smoke test: input (4, 751, 180) -> logits (4, 3)
       Trainable parameters: 291,203


### Cell 6: BiLSTM Model (`build_bilstm_model`)

In [6]:
# ============================================================================
# CELL 6: BiLSTM MODEL CONSTRUCTION & SMOKE TEST
# ============================================================================
def build_bilstm_model():
    """Return a fresh BiLSTMBaseline instance (input: [B, 751, 180] -> output: [B, 3])."""
    return BiLSTMBaseline(input_size=FEAT_COUNT, num_classes=NUM_CLASSES)

_bilstm_test = build_bilstm_model().eval()
_x_dummy     = torch.randn(4, W, FEAT_COUNT)
with torch.no_grad():
    _out_dummy = _bilstm_test(_x_dummy)

assert _out_dummy.shape == (4, NUM_CLASSES), f"Expected (4, {NUM_CLASSES}), got {_out_dummy.shape}"
assert torch.isfinite(_out_dummy).all(), "Non-finite logits in BiLSTM smoke test!"
print(f"[PASS] BiLSTMBaseline smoke test: input {tuple(_x_dummy.shape)} -> logits {tuple(_out_dummy.shape)}")
print(f"       Trainable parameters: {sum(p.numel() for p in _bilstm_test.parameters() if p.requires_grad):,}")
del _bilstm_test, _x_dummy, _out_dummy


[PASS] BiLSTMBaseline smoke test: input (4, 751, 180) -> logits (4, 3)
       Trainable parameters: 713,475


### Cell 7: BiLSTM-CNN Model (`build_bilstm_cnn_model`)

In [7]:
# ============================================================================
# CELL 7: BiLSTM-CNN MODEL CONSTRUCTION & ARCHITECTURE VERIFICATION
# ============================================================================
def build_bilstm_cnn_model():
    """Return a fresh BiLSTM_CNN_Baseline instance (input: [B, 751, 180] -> output: [B, 3])."""
    return BiLSTM_CNN_Baseline(input_size=FEAT_COUNT, num_classes=NUM_CLASSES)

_bilstm_cnn_test = build_bilstm_cnn_model().eval()
_x_dummy         = torch.randn(4, W, FEAT_COUNT)
with torch.no_grad():
    _out_dummy = _bilstm_cnn_test(_x_dummy)

assert _out_dummy.shape == (4, NUM_CLASSES), f"Expected (4, {NUM_CLASSES}), got {_out_dummy.shape}"
assert torch.isfinite(_out_dummy).all(), "Non-finite logits in BiLSTM-CNN smoke test!"

print("=" * 72)
print("BiLSTM-CNN Architecture Verification")
print("=" * 72)
print(f"  - BiLSTM-1 hidden_size : {_bilstm_cnn_test.lstm1.hidden_size} (bidirectional={_bilstm_cnn_test.lstm1.bidirectional}) ->LayerNorm {_bilstm_cnn_test.ln1.normalized_shape}")
print(f"  - Inter-layer Dropout  : p={_bilstm_cnn_test.lstm_dropout.p}")
print(f"  - BiLSTM-2 hidden_size : {_bilstm_cnn_test.lstm2.hidden_size} (bidirectional={_bilstm_cnn_test.lstm2.bidirectional}) ->LayerNorm {_bilstm_cnn_test.ln2.normalized_shape}")
print(f"  - Conv1D Block 1       : out_channels={_bilstm_cnn_test.conv1.out_channels}, kernel={_bilstm_cnn_test.conv1.kernel_size}, padding={_bilstm_cnn_test.conv1.padding}")
print(f"  - Conv1D Block 2       : out_channels={_bilstm_cnn_test.conv2.out_channels}, kernel={_bilstm_cnn_test.conv2.kernel_size}, padding={_bilstm_cnn_test.conv2.padding}")
print(f"  - Conv1D Block 3       : out_channels={_bilstm_cnn_test.conv3.out_channels}, kernel={_bilstm_cnn_test.conv3.kernel_size}, padding={_bilstm_cnn_test.conv3.padding}")
print(f"  - GlobalAvgPool1d      : output_size={_bilstm_cnn_test.global_pool.output_size}")
print(f"  - Dense Head           : {_bilstm_cnn_test.fc1.out_features} -> {_bilstm_cnn_test.fc2.out_features} -> {_bilstm_cnn_test.out.out_features}")
print(f"  - Total Parameters     : {sum(p.numel() for p in _bilstm_cnn_test.parameters()):,}")
print(f"  - Trainable Parameters : {sum(p.numel() for p in _bilstm_cnn_test.parameters() if p.requires_grad):,}")
print(f"[PASS] BiLSTM_CNN_Baseline smoke test: input {tuple(_x_dummy.shape)} -> logits {tuple(_out_dummy.shape)}")
print("=" * 72)
del _bilstm_cnn_test, _x_dummy, _out_dummy


BiLSTM-CNN Architecture Verification
  - BiLSTM-1 hidden_size : 128 (bidirectional=True) ->LayerNorm (256,)
  - Inter-layer Dropout  : p=0.3
  - BiLSTM-2 hidden_size : 64 (bidirectional=True) ->LayerNorm (128,)
  - Conv1D Block 1       : out_channels=64, kernel=(7,), padding=same
  - Conv1D Block 2       : out_channels=128, kernel=(5,), padding=same
  - Conv1D Block 3       : out_channels=256, kernel=(3,), padding=same
  - GlobalAvgPool1d      : output_size=1
  - Dense Head           : 128 -> 64 -> 3
  - Total Parameters     : 722,371
  - Trainable Parameters : 722,371
[PASS] BiLSTM_CNN_Baseline smoke test: input (4, 751, 180) -> logits (4, 3)


### Cell 8: Classical Logistic Regression Model (`build_classical_lr_model`)

In [8]:
# ============================================================================
# CELL 8: CLASSICAL LOGISTIC REGRESSION MODEL CONSTRUCTION & SMOKE TEST
# ============================================================================
def build_classical_lr_model():
    """Return a fresh scikit-learn LogisticRegression estimator."""
    return LogisticRegression(
        class_weight='balanced',
        max_iter=300,
        random_state=SEED,
        solver='lbfgs',
    )

_x_seq_dummy = np.random.randn(12, W, FEAT_COUNT).astype(np.float32)
_x_stat      = extract_statistical_features(_x_seq_dummy)
assert _x_stat.shape == (12, FEAT_COUNT * 2), f"Expected (12, {FEAT_COUNT * 2}), got {_x_stat.shape}"
_y_dummy     = np.array([0, 1, 2] * 4, dtype=int)
_lr_test     = build_classical_lr_model()
_lr_test.fit(_x_stat, _y_dummy)
_probs_dummy = _lr_test.predict_proba(_x_stat)
assert _probs_dummy.shape == (12, NUM_CLASSES), f"Expected (12, {NUM_CLASSES}), got {_probs_dummy.shape}"
print(f"[PASS] Classical_LR smoke test: seq {tuple(_x_seq_dummy.shape)} -> stat features {tuple(_x_stat.shape)} -> probs {tuple(_probs_dummy.shape)}")
del _x_seq_dummy, _x_stat, _y_dummy, _lr_test, _probs_dummy


[PASS] Classical_LR smoke test: seq (12, 751, 180) -> stat features (12, 360) -> probs (12, 3)


---
## Section C â€” Separate Model Training Cells
Each cell below independently trains and evaluates its designated model across all four preprocessing branches (`gsn_minmax`, `gsn_zscore`, `tbsn_minmax`, `tbsn_zscore`) and persists per-model and consolidated results immediately.

### Cell 9: Train CNN1D

In [9]:
# ============================================================================
# CELL 9: TRAIN CNN1D ACROSS ALL FOUR PREPROCESSING BRANCHES
# ============================================================================
cnn1d_results, cnn1d_history, cnn1d_predictions = train_deep_model_for_branches(
    model_name="CNN1D",
    model_factory=build_cnn1d_model,
    branches=EXPECTED_BRANCHES,
    fold_ids=EXPECTED_FOLDS,
    force_retrain=FORCE_RETRAIN_OTHER_MODELS,
)



STARTING DEEP MODEL TRAINING: CNN1D | Run ID: 20261012_115550_af9efd

----------------------------------------------------------------------------------------
Model                 : CNN1D
Branch                : gsn_minmax
Split / Fold          : trial_split
Partition Sizes       : TRAIN=619 | VALIDATION=131 | TEST=131
Device                : cuda
Optimizer             : Adam (lr=0.001, weight_decay=0.0)
Batch Size            : 32
Max Epochs / Patience : 100 / 10
Augmentation          : False
Scheduler             : ReduceLROnPlateau (monitors val_loss, factor=0.5)
Checkpoint Criterion  : Validation Loss (primary)
Run Mode              : FRESH (force_retrain=True)
Checkpoint Directory  : C:\Users\lab\Minor_project\Parkinson_gait\results\checkpoints\CNN1D\gsn_minmax\fold_trial_split\run_20261012_115550_af9efd
----------------------------------------------------------------------------------------
Epoch 001/100 | TrLoss: 1.1018 | VLoss: 1.0890 | TrAcc: 0.3506 | VAcc: 0.4275 | LR: 1.00e

### Cell 10: Train LSTM

In [10]:
# ============================================================================
# CELL 10: TRAIN LSTM ACROSS ALL FOUR PREPROCESSING BRANCHES
# ============================================================================
lstm_results, lstm_history, lstm_predictions = train_deep_model_for_branches(
    model_name="LSTM",
    model_factory=build_lstm_model,
    branches=EXPECTED_BRANCHES,
    fold_ids=EXPECTED_FOLDS,
    force_retrain=FORCE_RETRAIN_OTHER_MODELS,
)



STARTING DEEP MODEL TRAINING: LSTM | Run ID: 20261012_115946_484cb7

----------------------------------------------------------------------------------------
Model                 : LSTM
Branch                : gsn_minmax
Split / Fold          : trial_split
Partition Sizes       : TRAIN=619 | VALIDATION=131 | TEST=131
Device                : cuda
Optimizer             : Adam (lr=0.001, weight_decay=0.0)
Batch Size            : 32
Max Epochs / Patience : 100 / 10
Augmentation          : False
Scheduler             : ReduceLROnPlateau (monitors val_loss, factor=0.5)
Checkpoint Criterion  : Validation Loss (primary)
Run Mode              : FRESH (force_retrain=True)
Checkpoint Directory  : C:\Users\lab\Minor_project\Parkinson_gait\results\checkpoints\LSTM\gsn_minmax\fold_trial_split\run_20261012_115946_484cb7
----------------------------------------------------------------------------------------
Epoch 001/100 | TrLoss: 1.0816 | VLoss: 0.9988 | TrAcc: 0.4103 | VAcc: 0.5420 | LR: 1.00e-03

### Cell 11: Train BiLSTM

In [11]:
# ============================================================================
# CELL 11: TRAIN BiLSTM ACROSS ALL FOUR PREPROCESSING BRANCHES
# ============================================================================
bilstm_results, bilstm_history, bilstm_predictions = train_deep_model_for_branches(
    model_name="BiLSTM",
    model_factory=build_bilstm_model,
    branches=EXPECTED_BRANCHES,
    fold_ids=EXPECTED_FOLDS,
    force_retrain=FORCE_RETRAIN_OTHER_MODELS,
)



STARTING DEEP MODEL TRAINING: BiLSTM | Run ID: 20261012_120313_6e1e20

----------------------------------------------------------------------------------------
Model                 : BiLSTM
Branch                : gsn_minmax
Split / Fold          : trial_split
Partition Sizes       : TRAIN=619 | VALIDATION=131 | TEST=131
Device                : cuda
Optimizer             : Adam (lr=0.001, weight_decay=0.0)
Batch Size            : 32
Max Epochs / Patience : 100 / 10
Augmentation          : False
Scheduler             : ReduceLROnPlateau (monitors val_loss, factor=0.5)
Checkpoint Criterion  : Validation Loss (primary)
Run Mode              : FRESH (force_retrain=True)
Checkpoint Directory  : C:\Users\lab\Minor_project\Parkinson_gait\results\checkpoints\BiLSTM\gsn_minmax\fold_trial_split\run_20261012_120313_6e1e20
----------------------------------------------------------------------------------------
Epoch 001/100 | TrLoss: 1.0834 | VLoss: 1.0431 | TrAcc: 0.3926 | VAcc: 0.5267 | LR: 1.

### Cell 12: Train BiLSTM-CNN (Mandatory Dedicated Training Cell)

In [12]:
# ============================================================================
# CELL 12: DEDICATED BiLSTM-CNN TRAINING ACROSS ALL FOUR BRANCHES
# ============================================================================
# Configuration:
#   - Optimizer            : AdamW (lr=1e-4, weight_decay=0.0)
#   - Batch size           : 192
#   - Max epochs           : 400
#   - Early stop patience  : 10 (strictly on validation accuracy)
#   - Checkpoint criterion : Validation accuracy (with validation loss tie-break)
#   - Scheduler            : ReduceLROnPlateau monitoring validation loss
#   - Augmentation         : Disabled (False)
#   - Force fresh retrain  : FORCE_RETRAIN_BILSTM_CNN (defaults to True)
# ============================================================================
print("=" * 88)
print("DEDICATED BiLSTM-CNN TRAINING CELL")
print(f"FORCE_RETRAIN_BILSTM_CNN = {FORCE_RETRAIN_BILSTM_CNN}")
print("Branches: gsn_minmax, gsn_zscore, tbsn_minmax, tbsn_zscore")
print("=" * 88)

bilstm_cnn_results, bilstm_cnn_history, bilstm_cnn_predictions = train_deep_model_for_branches(
    model_name="BiLSTM_CNN",
    model_factory=build_bilstm_cnn_model,
    branches=["gsn_minmax", "gsn_zscore", "tbsn_minmax", "tbsn_zscore"],
    fold_ids=["trial_split"],
    force_retrain=FORCE_RETRAIN_BILSTM_CNN,
)

print("\n" + "=" * 88)
print("BiLSTM-CNN FOUR-BRANCH SUMMARY")
print("=" * 88)
print(
    f"{'Branch':<15} | {'FreshRun':<8} | {'EpochsRun':<9} | {'BestEp':<6} | "
    f"{'BestValAcc':<10} | {'BestValLoss':<11} | {'TestAcc':<8} | {'TestBAcc':<8} | "
    f"{'MacroF1':<8} | {'WeightF1':<8} | {'Kappa':<7} | {'MAE':<6}"
)
print("-" * 88)
for _r in bilstm_cnn_results:
    print(
        f"{_r['branch']:<15} | {str(_r['is_fresh_run']):<8} | "
        f"{_r['epochs_executed_this_run']:<9d} | {_r['best_epoch']:<6d} | "
        f"{_r['best_val_accuracy']:<10.4f} | {_r['best_val_loss']:<11.4f} | "
        f"{_r['test_accuracy']:<8.4f} | {_r['test_balanced_accuracy']:<8.4f} | "
        f"{_r['test_macro_f1']:<8.4f} | {_r['test_weighted_f1']:<8.4f} | "
        f"{_r['cohen_kappa']:<7.4f} | {_r['mae']:<6.4f}"
    )
print("=" * 88)


DEDICATED BiLSTM-CNN TRAINING CELL
FORCE_RETRAIN_BILSTM_CNN = True
Branches: gsn_minmax, gsn_zscore, tbsn_minmax, tbsn_zscore

STARTING DEEP MODEL TRAINING: BiLSTM_CNN | Run ID: 20261012_121151_76ae8e

----------------------------------------------------------------------------------------
Model                 : BiLSTM_CNN
Branch                : gsn_minmax
Split / Fold          : trial_split
Partition Sizes       : TRAIN=619 | VALIDATION=131 | TEST=131
Device                : cuda
Optimizer             : AdamW (lr=0.0001, weight_decay=0.0)
Batch Size            : 192
Max Epochs / Patience : 400 / 10
Augmentation          : False
Scheduler             : ReduceLROnPlateau (monitors val_loss, factor=0.5)
Checkpoint Criterion  : Validation Accuracy (primary) + Validation Loss (tie-breaker)
Run Mode              : FRESH (force_retrain=True)
Checkpoint Directory  : C:\Users\lab\Minor_project\Parkinson_gait\results\checkpoints\BiLSTM_CNN\gsn_minmax\fold_trial_split\run_20261012_121151_76ae8

### Cell 13: Train Classical Logistic Regression

In [13]:
# ============================================================================
# CELL 13: TRAIN CLASSICAL LOGISTIC REGRESSION ACROSS ALL FOUR BRANCHES
# ============================================================================
classical_lr_results, _, classical_lr_predictions = train_classical_lr_for_branches(
    model_factory=build_classical_lr_model,
    branches=EXPECTED_BRANCHES,
    fold_ids=EXPECTED_FOLDS,
    force_retrain=FORCE_RETRAIN_OTHER_MODELS,
)



STARTING CLASSICAL BASELINE TRAINING: Classical_LR | Run ID: 20261012_121651_984e66

HELD-OUT TEST EVALUATION â€” Classical_LR | gsn_minmax | trial_split
Run Status               : FRESHLY TRAINED
Partition Samples        : TRAIN=619, VAL=131, TEST=131
LBFGS Iterations         : 186
Validation Accuracy      : 0.9542 (Val LogLoss: 0.2499)
TEST Accuracy            : 0.9542
TEST Balanced Accuracy   : 0.9514
TEST Macro F1            : 0.9548
TEST Weighted F1         : 0.9545
TEST Cohen's Kappa       : 0.9292
TEST MAE                 : 0.0458

Confusion Matrix (rows=true, cols=pred):
[[54  2  0]
 [ 2 42  0]
 [ 0  2 29]]

Classification Report:
              precision    recall  f1-score   support

     Class 0     0.9643    0.9643    0.9643        56
     Class 1     0.9130    0.9545    0.9333        44
     Class 2     1.0000    0.9355    0.9667        31

    accuracy                         0.9542       131
   macro avg     0.9591    0.9514    0.9548       131
weighted avg     0.9555   

---
## Section D â€” Combined Results and Completion Audit
### Cell 14: Combined Results Summary and Audit

In [14]:
# ============================================================================
# CELL 14: COMBINED RESULTS AND COMPLETION AUDIT
# ============================================================================
print("=" * 100)
print("NOTEBOOK 11 â€” COMBINED BASELINE RESULTS & COMPLETION AUDIT")
print("=" * 100)

# Gather results from current session first, falling back to per-model saved CSVs if a cell was not run in this session
combined_metrics_rows = []
for m_name in EXPECTED_MODELS:
    if m_name in SESSION_MODEL_RESULTS:
        combined_metrics_rows.extend(SESSION_MODEL_RESULTS[m_name])
    else:
        per_model_csv = PROJECT_ROOT / 'results' / 'tables' / f"{m_name}_test_metrics.csv"
        if per_model_csv.exists():
            df_m = pd.read_csv(per_model_csv)
            # Loaded from disk without running in this session -> mark is_fresh_run=False
            df_m['is_fresh_run'] = False
            df_m['run_mode'] = 'DISK_LOADED'
            combined_metrics_rows.extend(df_m.to_dict(orient='records'))

if len(combined_metrics_rows) == 0:
    raise RuntimeError("No model results found in session or in results/tables/!")

m_df = pd.DataFrame(combined_metrics_rows)
m_df.to_csv(PROJECT_ROOT / 'results' / 'tables' / 'baseline_final_test_metrics.csv', index=False)

# Verify expected model x branch x fold combinations
expected_combos = {
    (m, b, f)
    for m in EXPECTED_MODELS
    for b in EXPECTED_BRANCHES
    for f in EXPECTED_FOLDS
}

completed_combos = set()
freshly_trained_combos = set()
reused_combos = set()
bilstm_cnn_fresh_branches = set()

for r in combined_metrics_rows:
    combo = (str(r['model']), str(r['branch']), str(r['fold']))
    ckpt_exists = Path(str(r.get('checkpoint_path', ''))).exists()
    if ckpt_exists:
        completed_combos.add(combo)
        if bool(r.get('is_fresh_run', False)) and int(r.get('epochs_executed_this_run', 0)) > 0:
            freshly_trained_combos.add(combo)
            if r['model'] == 'BiLSTM_CNN':
                bilstm_cnn_fresh_branches.add(str(r['branch']))
        else:
            reused_combos.add(combo)

missing_combos = sorted(expected_combos - completed_combos)

# Display branch-level summary table
display_cols = [
    'model', 'branch', 'fold', 'run_mode', 'is_fresh_run',
    'n_train', 'n_val', 'n_test',
    'epochs_executed_this_run', 'best_epoch',
    'best_val_accuracy', 'best_val_loss',
    'test_accuracy', 'test_balanced_accuracy',
    'test_macro_f1', 'test_weighted_f1', 'cohen_kappa', 'mae',
]
available_cols = [c for c in display_cols if c in m_df.columns]
print("\nBRANCH-LEVEL EVALUATION METRICS:")
print(m_df[available_cols].to_string(index=False))

print("\n" + "-" * 100)
print("AUDIT SUMMARY:")
print(f"  Expected (model, branch, split) combinations : {len(expected_combos)}")
print(f"  Completed combinations with valid checkpoint : {len(completed_combos)}")
print(f"  Freshly trained in this session              : {len(freshly_trained_combos)}")
print(f"  Reused from previous checkpoints             : {len(reused_combos)}")
print(f"  Missing combinations                         : {len(missing_combos)}")
print(f"  BiLSTM-CNN branches freshly trained          : {sorted(bilstm_cnn_fresh_branches)} ({len(bilstm_cnn_fresh_branches)}/{len(EXPECTED_BRANCHES)})")

if missing_combos:
    print("\n  [!] Missing combinations:")
    for mc in missing_combos:
        print(f"      - {mc}")

# Strict readiness check:
# 1. All expected (model, branch, fold) combinations have valid checkpoints and test metrics.
# 2. All 4 BiLSTM-CNN branches were freshly trained in the current run (recorded actual epochs > 0).
all_combos_complete = (len(missing_combos) == 0) and (len(completed_combos) == len(expected_combos))
bilstm_cnn_fresh_ok = (bilstm_cnn_fresh_branches == set(EXPECTED_BRANCHES))

print("-" * 100)
if all_combos_complete and bilstm_cnn_fresh_ok:
    print("FINAL STATUS: READY")
else:
    print("FINAL STATUS: INCOMPLETE")
    if not all_combos_complete:
        print(f"  Reason: {len(missing_combos)} model/branch combinations are missing.")
    if not bilstm_cnn_fresh_ok:
        missing_bilstm = sorted(set(EXPECTED_BRANCHES) - bilstm_cnn_fresh_branches)
        print(f"  Reason: BiLSTM-CNN fresh training not yet executed for branches: {missing_bilstm}")
print("=" * 100)


NOTEBOOK 11 â€” COMBINED BASELINE RESULTS & COMPLETION AUDIT

BRANCH-LEVEL EVALUATION METRICS:
       model      branch        fold run_mode  is_fresh_run  n_train  n_val  n_test  epochs_executed_this_run  best_epoch  best_val_accuracy  best_val_loss  test_accuracy  test_balanced_accuracy  test_macro_f1  test_weighted_f1  cohen_kappa      mae
       CNN1D  gsn_minmax trial_split    FRESH          True      619    131     131                        74          64           1.000000       0.000535       0.977099                0.972542       0.976294          0.976989     0.964563 0.038168
       CNN1D  gsn_zscore trial_split    FRESH          True      619    131     131                        15           5           0.992366       0.052779       0.984733                0.978495       0.981481          0.984648     0.976401 0.015267
       CNN1D tbsn_minmax trial_split    FRESH          True      619    131     131                       100         100           0.900763       0.245107